In [2]:
import requests
import pandas as pd

API_KEY = "9b63f277849b43a24a78728d2dd39486"

SPORTS = {
    # US Sports
    "NFL": "americanfootball_nfl",
    "NBA": "basketball_nba",
    "MLB": "baseball_mlb",
    "NHL": "icehockey_nhl",
    "MMA": "mma_mixed_martial_arts",
    "Boxing": "boxing_boxing",

    # Soccer
    "EPL": "soccer_epl",
    "La Liga": "soccer_spain_la_liga",
    "Serie A": "soccer_italy_serie_a",
    "Bundesliga": "soccer_germany_bundesliga",
    "Ligue 1": "soccer_france_ligue_one",
    "Champions League": "soccer_uefa_champs_league",
    "MLS": "soccer_usa_mls",
}

BOOKMAKERS = "draftkings,fanduel,betmgm,espnbet,ballybet"

# Files saved directly into repo folder
OUTPUT_EXCEL = "Arbitrage_Input.xlsx"
OUTPUT_CSV = "Arbitrage_Input.csv"

# Markets to pull
MARKETS = ["h2h", "spreads", "totals"]


def fetch_odds(sport_key):

    url = f"https://api.the-odds-api.com/v4/sports/{sport_key}/odds"

    params = {
        "apiKey": API_KEY,
        "regions": "us",
        "markets": ",".join(MARKETS),
        "oddsFormat": "american",
        "bookmakers": BOOKMAKERS
    }

    response = requests.get(url, params=params)

    response.raise_for_status()

    return response.json()


def main():

    all_rows = []

    for sport_name, sport_key in SPORTS.items():

        print(f"Fetching {sport_name} odds...")

        try:
            data = fetch_odds(sport_key)

        except Exception as e:
            print(f"Error fetching {sport_name}: {e}")
            continue

        for event in data:

            event_name = f"{event['home_team']} vs {event['away_team']}"

            for bookmaker in event.get("bookmakers", []):

                book = bookmaker["title"]

                for market in bookmaker.get("markets", []):

                    market_key = market["key"]

                    if market_key not in MARKETS:
                        continue

                    outcomes = market.get("outcomes", [])

                    if not outcomes:
                        continue

                    # Get spread/total line
                    line = None

                    if market_key in ["spreads", "totals"]:
                        line = outcomes[0].get("point")

                    market_str = (
                        f"{market_key}"
                        + (f"_{line}" if line is not None else "")
                    )

                    # Market integrity checks

                    # Soccer moneyline should have 3 outcomes
                    if "soccer" in sport_key and market_key == "h2h":
                        if len(outcomes) != 3:
                            continue

                    # US sports moneyline should have 2 outcomes
                    if "soccer" not in sport_key and market_key == "h2h":
                        if len(outcomes) != 2:
                            continue

                    for outcome in outcomes:

                        all_rows.append({
                            "Sport": sport_name,
                            "Event": event_name,
                            "Market": market_str,
                            "Outcome": outcome["name"],
                            "Bookmaker": book,
                            "Odds": outcome["price"]
                        })

    if all_rows:

        df = pd.DataFrame(all_rows)

        # Sort for readability
        df = df.sort_values(
            by=["Sport", "Event", "Market", "Outcome", "Bookmaker"]
        )

        # Save Excel
        df.to_excel(OUTPUT_EXCEL, index=False)

        # Save CSV
        df.to_csv(OUTPUT_CSV, index=False)

        print("\n==============================")
        print("ODDS SCRAPE COMPLETE")
        print("==============================")
        print(f"Excel file updated: {OUTPUT_EXCEL}")
        print(f"CSV file updated: {OUTPUT_CSV}")
        print(f"Total rows written: {len(df)}")

    else:
        print("No odds fetched. Check API key, quota, or bookmaker availability.")


if __name__ == "__main__":
    main()

Fetching NFL odds...
Fetching NBA odds...
Fetching MLB odds...
Fetching NHL odds...
Fetching MMA odds...
Fetching Boxing odds...
Fetching EPL odds...
Fetching La Liga odds...
Fetching Serie A odds...
Fetching Bundesliga odds...
Fetching Ligue 1 odds...
Error fetching Ligue 1: 429 Client Error: Too Many Requests for url: https://api.the-odds-api.com/v4/sports/soccer_france_ligue_one/odds?apiKey=9b63f277849b43a24a78728d2dd39486&regions=us&markets=h2h%2Cspreads%2Ctotals&oddsFormat=american&bookmakers=draftkings%2Cfanduel%2Cbetmgm%2Cespnbet%2Cballybet
Fetching Champions League odds...
Error fetching Champions League: 429 Client Error: Too Many Requests for url: https://api.the-odds-api.com/v4/sports/soccer_uefa_champs_league/odds?apiKey=9b63f277849b43a24a78728d2dd39486&regions=us&markets=h2h%2Cspreads%2Ctotals&oddsFormat=american&bookmakers=draftkings%2Cfanduel%2Cbetmgm%2Cespnbet%2Cballybet
Fetching MLS odds...

ODDS SCRAPE COMPLETE
Excel file updated: Arbitrage_Input.xlsx
CSV file updated